In [ ]:
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd

import os
import matplotlib.pyplot as plt
from glob import glob
import seaborn as sns 
from tqdm.notebook import tqdm

In [ ]:
energyDF=pd.read_csv("../../../classical/energies.csv",index_col=0)

In [ ]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [ ]:
moldf.shape

In [ ]:
import re

def process_list(data_list):
    """
    Processes a list of strings to perform the following operations:
    1.  Strips trailing newline characters.
    2.  Parses the 'LUCJ' string into three separate elements.
    3.  Converts the final element from a string to a float.
    """
    processed_list = []
    for item in data_list:
        item = item.strip()

        if "LUCJ" in item:
            # Use a regular expression to extract the components
            match = re.search(r'(LUCJ)\(L=(.*?)\)/(.*)', item)
            if match:
                processed_list.extend(match.groups())
            else:
                processed_list.append(item)
        else:
            processed_list.append(item)
    
    # Convert the last element to a float
    # We use a try-except block in case the last element is not a number
    try:
        processed_list[-1] = float(processed_list[-1])
    except (ValueError, IndexError):
        pass # The last element is not a float, so we ignore it

    return processed_list

In [ ]:
postprocessed = []
for i in tqdm(glob("./energies/*txt"),desc='Running'):
    with open(i,'r') as f:
        lines = f.readlines()

            
    moldict = pd.DataFrame.from_dict(dict(zip(["Basis Set","Molecule","Method","L","Injected","Energy"],process_list(lines))),orient='index').T
    postprocessed.append(moldict)

In [ ]:
LUCJDF=pd.concat(postprocessed).reset_index().drop(columns=['index']).astype({"L":int,"Energy":float})
LUCJDF['Method'] =[f"LUCJ(L={i})" for i in LUCJDF['L']]

In [ ]:
for a,b in moldf[['molecule','mol_filename']].values:
    if 'GDB' in b:
        name = b.replace('.xyz','')
        energyDF['Molecule'] = energyDF['Molecule'].replace(name,a)

In [ ]:
energyDF['L'] = len(energyDF)*[np.nan]
energyDF['Injected'] = len(energyDF)*[np.nan]

In [ ]:
upDF = pd.concat([energyDF,LUCJDF]).reset_index().drop(columns=['index'])
upDF['Deviation'] = np.nan

In [ ]:
uniqueBasis = energyDF['Basis Set'].unique()
uniqueMol = energyDF['Molecule'].unique()
uniqueInj = energyDF['Injected'].unique()
uniqueLayers = upDF['L'].unique()


In [ ]:
uniqueMol

In [ ]:
for mol in uniqueMol:
    for basis in uniqueBasis:
        # Get the CASCI reference energy (scalar)
        casci_row = upDF.loc[
            (upDF["Molecule"] == mol)
            & (upDF["Method"] == "CCSD")
            & (upDF["Basis Set"] == basis),
            "Energy",
        ]
        if casci_row.empty:
            continue  # skip if no CASCI reference for this (mol, basis)
        casci_e = casci_row.values[0]  # extract scalar

        # Compute deviations for other methods with the same mol/basis
        mask = (
            (upDF["Molecule"] == mol)
            & (upDF["Basis Set"] == basis)
            & (upDF["Method"] != "CCSD")
        )
        upDF.loc[mask, "Deviation"] = (upDF.loc[mask, "Energy"] - casci_e)*1e3


In [ ]:
sns.relplot(data=upDF, x="total_bill", y="tip", hue="day", col="time", row="sex")


In [ ]:
# Create the FacetGrid with bar plots
g = sns.catplot(
    data=upDF,
    row="Molecule",
    y="Deviation",
    x="Method",
    hue="Basis Set",
    col="Injected",
    kind="bar",
    height=4,
    aspect=0.6,
)

# Compute global deviation range for consistent shading/scaling
ymin = upDF["Deviation"].min()
ymax = upDF["Deviation"].max()
pad = 0.1 * max(abs(ymin), abs(ymax))

# Apply styling to **all subplots**
for ax in g.axes.flat:
    ax.set_yscale("symlog", linthresh=1e-11)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.axhline(0, color="gray", lw=1, ls="--", alpha=0.8)
    
    # Gray fill region across same y-span
    ax.fill_between(
        np.linspace(-0.5, len(upDF["Method"].unique()) - 0.5, 100),
        -1.6, 1.6,
        color="gray", alpha=0.2
    )

    ax.set_xlim(-0.5, len(upDF["Method"].unique()) - 0.5)

    # Rotate and align x tick labels
    for label in ax.get_xticklabels():
        label.set_rotation(90)
        label.set_horizontalalignment("center")
        label.set_verticalalignment("top")

# Adjust titles and axis labels
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.set_axis_labels("Method", "Deviation")

# Move legend outside the main plot (right side)
g._legend.set_bbox_to_anchor((1.05, 0.5))
g._legend.set_frame_on(False)
plt.subplots_adjust(right=0.85)  # make space for legend

plt.tight_layout()
plt.show()


In [ ]:
# g = sns.FacetGrid(upDF, col="Basis Set",row='Molecule')
# g.map_dataframe(sns.barplot, x="Method",y='Energy',palette="Paired")

In [ ]:
# # This creates a boolean mask that is True for every row where 'Method' contains "LUCJ"
# mask = upDF['Method'].str.contains("LUCJ", case=False, na=False)

# # Use the boolean mask to filter the DataFrame
# filtered_df = upDF[mask]


In [ ]:

# devDF = upDF[(upDF['Basis Set']=='STO-3G')&(upDF['Molecule']=='ammonia')]
upDF['Deviation'] = (upDF['Energy'] - upDF.loc[upDF['Method'] == 'CASCI', 'Energy'].values[0])*1e3

In [ ]:
# upDF.loc[(upDF['Molecule']=='formaldehyde')&(upDF['Basis Set']=='STO-3G'),:].sort_values(by=['Method','L','Injected'])

In [ ]:
upDF.sort_values(by=['Method','L','Injected','Deviation'])

In [ ]:
sns.barplot(devDF,x='Method',y='Deviation')
# plt.yscale('symlog')
# plt.ylim(-1e-2,1e2)
